In [10]:
import pandas as pd
import numpy as np
import json
import time
import os
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.graph.GraphNode import GraphNode

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

gene_burden = pd.read_csv(os.path.join(out_dir, "gene_burden_matrix_signed_protein_coding.csv"), index_col=0)
results_df = pd.read_csv(os.path.join(out_dir, "gene_doubleml_stability_smoking_AA.csv"))
shortlist_100 = results_df[results_df["stability_fraction"] == 1.0].copy()
shortlist_dedup = shortlist_100[shortlist_100["gene"] != "TAS1R3"]

meta_df = pd.read_csv(os.path.join(out_dir, "checkpoint2b_metadata_relatedness_filtered.csv"))
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
meta_df = meta_df.set_index("sample_id")
meta_df["smoking_status_bin"] = (meta_df["smoking_status"] == "Smoker").astype(int)

sample_cols = gene_burden.columns.tolist()
pheno = meta_df.loc[sample_cols, "smoking_status_bin"].astype(float)

gene_burden_final = gene_burden.loc[shortlist_dedup["gene"]]
X_genes_pc = gene_burden_final.T.values
Y_pc = pheno.values.reshape(-1, 1)
X_pc_full2 = np.hstack([X_genes_pc, Y_pc])
col_names2 = gene_burden_final.index.tolist() + ["smoking_status"]

print("Ready:", X_pc_full2.shape, len(col_names2))

Ready: (3036, 66) 66


In [11]:
start = time.time()
cg_free = pc(
    data=X_pc_full2,
    alpha=0.001,
    indep_test=fisherz,
    stable=True,
    uc_rule=0,
    uc_priority=2,
    depth=3,
    verbose=False,
    show_progress=True,
    node_names=col_names2
)
print(f"Completed in {time.time()-start:.1f}s")

KeyboardInterrupt: 

In [12]:
import time

test_sizes = [10, 20, 30, 40, 50]
timings = {}

for size in test_sizes:
    cols_subset = col_names2[:size] + ["smoking_status"] if "smoking_status" not in col_names2[:size] else col_names2[:size]
    idx_subset = [col_names2.index(c) for c in cols_subset]
    X_subset = X_pc_full2[:, idx_subset]

    n = len(cols_subset)
    outcome_idx = cols_subset.index("smoking_status")

    bk = BackgroundKnowledge()
    nodes = [GraphNode(name) for name in cols_subset]
    for i in range(n - 1):
        bk.add_node_to_tier(nodes[i], 0)
    bk.add_node_to_tier(nodes[outcome_idx], 1)

    start = time.time()
    cg = pc(
        data=X_subset,
        alpha=0.001,
        indep_test=fisherz,
        stable=True,
        uc_rule=0,
        uc_priority=2,
        background_knowledge=bk,
        depth=3,
        verbose=False,
        show_progress=False,
        node_names=cols_subset
    )
    elapsed = time.time() - start
    timings[size] = elapsed
    print(f"{size} nodes: {elapsed:.1f}s")

print("\nFull timing summary:", timings)

10 nodes: 0.2s
20 nodes: 6.7s
30 nodes: 36.8s
40 nodes: 490.9s


KeyboardInterrupt: 

In [13]:
import numpy as np

n_groups = 4  # ~16 genes per group, well within the fast zone (30 nodes = 37s)
genes_list = shortlist_dedup.sort_values("stability_fraction", ascending=False)["gene"].tolist()

groups = [genes_list[i::n_groups] for i in range(n_groups)]
for idx, g in enumerate(groups):
    print(f"Group {idx+1}: {len(g)} genes -> {g}")

Group 1: 17 genes -> ['F10', 'STIL', 'PCDH12', 'DVL1', 'DNAH5', 'LIMS2', 'NLRP8', 'USH2A', 'TICAM1', 'CELSR2', 'PLEC', 'FRAS1', 'IGSF5', 'CHD6', 'ZNF418', 'PLXNA2', 'AXDND1']
Group 2: 16 genes -> ['FREM2', 'DNAH11', 'LRP2', 'ASTN1', 'HYDIN', 'SPEF2', 'LOXHD1', 'BCLAF1', 'LZTFL1', 'HMCN1', 'UTP20', 'DLEC1', 'CNTRL', 'C4orf22', 'COL6A6', 'OR51I1']
Group 3: 16 genes -> ['ZNF805', 'FAM179A', 'KIAA1377', 'LAMA5', 'DCHS2', 'ATG2A', 'CP', 'TEP1', 'TRIM45', 'CENPF', 'RYR1', 'CEP135', 'XIRP1', 'RNF213', 'CHPF2', 'FREM1']
Group 4: 16 genes -> ['DNAH1', 'FAM126A', 'GPR98', 'FBN3', 'SIGLEC1', 'EXO1', 'IL19', 'TG', 'TSC2', 'RTN4', 'CLIP1', 'PET112', 'CCBL2', 'MDN1', 'CPAMD8', 'LAMA3']


In [14]:
import numpy as np
import time
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.graph.GraphNode import GraphNode

all_direct_parents = {}
all_results = {}

for group_idx, gene_group in enumerate(groups):
    group_cols = gene_group + ["smoking_status"]
    idx_subset = [col_names2.index(c) for c in group_cols]
    X_subset = X_pc_full2[:, idx_subset]

    n = len(group_cols)
    outcome_idx = group_cols.index("smoking_status")

    bk = BackgroundKnowledge()
    nodes = [GraphNode(name) for name in group_cols]
    for i in range(n - 1):
        bk.add_node_to_tier(nodes[i], 0)
    bk.add_node_to_tier(nodes[outcome_idx], 1)

    start = time.time()
    cg = pc(
        data=X_subset,
        alpha=0.001,
        indep_test=fisherz,
        stable=True,
        uc_rule=0,
        uc_priority=2,
        background_knowledge=bk,
        depth=3,
        verbose=False,
        show_progress=False,
        node_names=group_cols
    )
    elapsed = time.time() - start
    print(f"Group {group_idx+1} ({n} nodes): {elapsed:.1f}s")

    adj = cg.G.graph
    direct_parents = []
    for i in range(n):
        if group_cols[i] == "smoking_status":
            continue
        j = outcome_idx
        if (adj[i, j] == -1 and adj[j, i] == 1) or (adj[i, j] == -1 and adj[j, i] == -1):
            direct_parents.append(group_cols[i])

    print(f"  Direct parents found: {direct_parents}")
    all_direct_parents[f"group_{group_idx+1}"] = direct_parents
    all_results[f"group_{group_idx+1}"] = cg

print("\n--- Summary ---")
total_parents = []
for g, parents in all_direct_parents.items():
    total_parents.extend(parents)
print(f"Total direct parents across all groups: {len(total_parents)}")
print(total_parents)

Group 1 (18 nodes): 2.7s
  Direct parents found: ['DVL1', 'DNAH5', 'LIMS2', 'USH2A', 'TICAM1', 'CELSR2', 'PLEC', 'FRAS1', 'IGSF5', 'CHD6', 'ZNF418', 'PLXNA2']
Group 2 (17 nodes): 2.4s
  Direct parents found: ['FREM2', 'ASTN1', 'HYDIN', 'SPEF2', 'LZTFL1', 'HMCN1', 'UTP20', 'DLEC1', 'CNTRL', 'COL6A6']
Group 3 (17 nodes): 4.5s
  Direct parents found: ['ZNF805', 'LAMA5', 'DCHS2', 'TEP1', 'TRIM45', 'CENPF', 'RYR1', 'CEP135', 'XIRP1', 'RNF213', 'CHPF2', 'FREM1']
Group 4 (17 nodes): 7.6s
  Direct parents found: ['DNAH1', 'FAM126A', 'GPR98', 'SIGLEC1', 'EXO1', 'TSC2', 'RTN4', 'CLIP1', 'PET112', 'CCBL2', 'MDN1', 'CPAMD8', 'LAMA3']

--- Summary ---
Total direct parents across all groups: 47
['DVL1', 'DNAH5', 'LIMS2', 'USH2A', 'TICAM1', 'CELSR2', 'PLEC', 'FRAS1', 'IGSF5', 'CHD6', 'ZNF418', 'PLXNA2', 'FREM2', 'ASTN1', 'HYDIN', 'SPEF2', 'LZTFL1', 'HMCN1', 'UTP20', 'DLEC1', 'CNTRL', 'COL6A6', 'ZNF805', 'LAMA5', 'DCHS2', 'TEP1', 'TRIM45', 'CENPF', 'RYR1', 'CEP135', 'XIRP1', 'RNF213', 'CHPF2', 'FREM1'

In [15]:
import random
random.seed(99)
genes_shuffled = genes_list.copy()
random.shuffle(genes_shuffled)
groups_v2 = [genes_shuffled[i::4] for i in range(4)]

all_direct_parents_v2 = {}
for group_idx, gene_group in enumerate(groups_v2):
    group_cols = gene_group + ["smoking_status"]
    idx_subset = [col_names2.index(c) for c in group_cols]
    X_subset = X_pc_full2[:, idx_subset]
    n = len(group_cols)
    outcome_idx = group_cols.index("smoking_status")

    bk = BackgroundKnowledge()
    nodes = [GraphNode(name) for name in group_cols]
    for i in range(n - 1):
        bk.add_node_to_tier(nodes[i], 0)
    bk.add_node_to_tier(nodes[outcome_idx], 1)

    cg = pc(data=X_subset, alpha=0.001, indep_test=fisherz, stable=True,
            uc_rule=0, uc_priority=2, background_knowledge=bk, depth=3,
            verbose=False, show_progress=False, node_names=group_cols)

    adj = cg.G.graph
    direct_parents = [group_cols[i] for i in range(n) if group_cols[i] != "smoking_status"
                       and ((adj[i, outcome_idx] == -1 and adj[outcome_idx, i] == 1)
                            or (adj[i, outcome_idx] == -1 and adj[outcome_idx, i] == -1))]
    all_direct_parents_v2[f"group_{group_idx+1}"] = direct_parents

total_v2 = [g for parents in all_direct_parents_v2.values() for g in parents]
print("Total direct parents (2nd random split):", len(total_v2))

overlap = set(total_parents) & set(total_v2)
print(f"\nOverlap between the two splits: {len(overlap)} / {len(set(total_parents) | set(total_v2))} genes")
print("Genes stable across BOTH splits:", sorted(overlap))

Total direct parents (2nd random split): 47

Overlap between the two splits: 41 / 53 genes
Genes stable across BOTH splits: ['ASTN1', 'CCBL2', 'CELSR2', 'CENPF', 'CEP135', 'CHD6', 'CHPF2', 'CLIP1', 'CNTRL', 'COL6A6', 'CPAMD8', 'DCHS2', 'DLEC1', 'DNAH1', 'DNAH5', 'DVL1', 'EXO1', 'FRAS1', 'FREM1', 'FREM2', 'GPR98', 'HMCN1', 'HYDIN', 'LAMA3', 'LAMA5', 'LZTFL1', 'MDN1', 'PLEC', 'PLXNA2', 'RNF213', 'RTN4', 'RYR1', 'SIGLEC1', 'SPEF2', 'TICAM1', 'TSC2', 'USH2A', 'UTP20', 'XIRP1', 'ZNF418', 'ZNF805']


In [16]:
import numpy as np
import time
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.graph.GraphNode import GraphNode

n_groups_2 = 2
groups_2 = [genes_list[i::n_groups_2] for i in range(n_groups_2)]
for idx, g in enumerate(groups_2):
    print(f"Group {idx+1}: {len(g)} genes")

all_direct_parents_2grp = {}

for group_idx, gene_group in enumerate(groups_2):
    group_cols = gene_group + ["smoking_status"]
    idx_subset = [col_names2.index(c) for c in group_cols]
    X_subset = X_pc_full2[:, idx_subset]

    n = len(group_cols)
    outcome_idx = group_cols.index("smoking_status")

    bk = BackgroundKnowledge()
    nodes = [GraphNode(name) for name in group_cols]
    for i in range(n - 1):
        bk.add_node_to_tier(nodes[i], 0)
    bk.add_node_to_tier(nodes[outcome_idx], 1)

    start = time.time()
    cg = pc(
        data=X_subset,
        alpha=0.001,
        indep_test=fisherz,
        stable=True,
        uc_rule=0,
        uc_priority=2,
        background_knowledge=bk,
        depth=3,
        verbose=False,
        show_progress=False,
        node_names=group_cols
    )
    elapsed = time.time() - start
    print(f"Group {group_idx+1} ({n} nodes): {elapsed:.1f}s")

    adj = cg.G.graph
    direct_parents = [group_cols[i] for i in range(n) if group_cols[i] != "smoking_status"
                       and ((adj[i, outcome_idx] == -1 and adj[outcome_idx, i] == 1)
                            or (adj[i, outcome_idx] == -1 and adj[outcome_idx, i] == -1))]
    print(f"  Direct parents found: {direct_parents}")
    all_direct_parents_2grp[f"group_{group_idx+1}"] = direct_parents

total_2grp = [g for parents in all_direct_parents_2grp.values() for g in parents]
print(f"\nTotal direct parents (2-group split): {len(total_2grp)}")

# Compare against the stable-41 list from the 4-group consistency check
overlap_with_41 = set(total_2grp) & set(overlap)
print(f"Overlap with the 41-gene stable list: {len(overlap_with_41)} / {len(set(total_2grp) | set(overlap))}")
print("Genes in 2-group result but NOT in stable-41:", set(total_2grp) - set(overlap))
print("Genes in stable-41 but NOT in 2-group result:", set(overlap) - set(total_2grp))

Group 1: 33 genes
Group 2: 32 genes
Group 1 (34 nodes): 178.3s
  Direct parents found: ['ZNF805', 'DVL1', 'LAMA5', 'DNAH5', 'DCHS2', 'USH2A', 'TICAM1', 'CELSR2', 'PLEC', 'RYR1', 'FRAS1', 'XIRP1', 'CHD6', 'RNF213', 'ZNF418', 'FREM1']
Group 2 (33 nodes): 235.4s
  Direct parents found: ['FREM2', 'DNAH1', 'GPR98', 'ASTN1', 'HYDIN', 'SIGLEC1', 'EXO1', 'LZTFL1', 'TSC2', 'HMCN1', 'RTN4', 'UTP20', 'CNTRL', 'CCBL2', 'MDN1', 'COL6A6', 'CPAMD8']

Total direct parents (2-group split): 33
Overlap with the 41-gene stable list: 33 / 41
Genes in 2-group result but NOT in stable-41: set()
Genes in stable-41 but NOT in 2-group result: {'DLEC1', 'LAMA3', 'CEP135', 'CHPF2', 'CLIP1', 'SPEF2', 'PLXNA2', 'CENPF'}


In [17]:
import requests
import json
import pandas as pd

genes_33 = ['ZNF805', 'DVL1', 'LAMA5', 'DNAH5', 'DCHS2', 'USH2A', 'TICAM1', 'CELSR2', 
            'PLEC', 'RYR1', 'FRAS1', 'XIRP1', 'CHD6', 'RNF213', 'ZNF418', 'FREM1',
            'FREM2', 'DNAH1', 'GPR98', 'ASTN1', 'HYDIN', 'SIGLEC1', 'EXO1', 'LZTFL1', 
            'TSC2', 'HMCN1', 'RTN4', 'UTP20', 'CNTRL', 'CCBL2', 'MDN1', 'COL6A6', 'CPAMD8']

# Submit gene list to Enrichr
enrichr_url = 'https://maayanlab.cloud/Enrichr/addList'
genes_str = '\n'.join(genes_33)
payload = {'list': (None, genes_str), 'description': (None, 'AA_smoking_gene_burden_33')}
response = requests.post(enrichr_url, files=payload)
response.raise_for_status()
user_list_id = response.json()['userListId']
print("Submitted, list ID:", user_list_id)

# Query enrichment against a couple of relevant libraries
libraries = ['GO_Biological_Process_2023', 'KEGG_2021_Human', 'Reactome_2022']

enrichment_results = {}
for lib in libraries:
    query_url = f'https://maayanlab.cloud/Enrichr/enrich?userListId={user_list_id}&backgroundType={lib}'
    r = requests.get(query_url)
    r.raise_for_status()
    data = r.json()[lib]
    df = pd.DataFrame(data, columns=['Rank', 'Term', 'P-value', 'Z-score', 'Combined score',
                                       'Genes', 'Adjusted P-value', 'Old P-value', 'Old adjusted P-value'])
    enrichment_results[lib] = df
    print(f"\n--- {lib} (top 10 by adjusted p-value) ---")
    print(df.sort_values('Adjusted P-value').head(10)[['Term', 'P-value', 'Adjusted P-value', 'Genes']].to_string(index=False))

Submitted, list ID: 133304177

--- GO_Biological_Process_2023 (top 10 by adjusted p-value) ---
                                                                        Term  P-value  Adjusted P-value                 Genes
                                                Cilium Movement (GO:0003341) 0.000125          0.036852 [DNAH1, DNAH5, HYDIN]
            Wnt Signaling Pathway, Planar Cell Polarity Pathway (GO:0060071) 0.000837          0.076680        [DVL1, CELSR2]
                 Regulation Of Establishment Of Planar Polarity (GO:0090175) 0.001042          0.076680        [DVL1, CELSR2]
                                            Neural Tube Closure (GO:0001843) 0.001270          0.076680          [DVL1, TSC2]
                                         Dendrite Morphogenesis (GO:0048813) 0.001518          0.076680        [DVL1, CELSR2]
                               Axonemal Dynein Complex Assembly (GO:0070286) 0.001696          0.076680        [DNAH1, DNAH5]
                       